# 💊 PharmaChain — Google Colab Launcher

**Blockchain-based Pharmaceutical Supply Chain Tracking System**

This notebook sets up and launches PharmaChain entirely inside Google Colab.
It installs .NET 6.0, clones the project, configures SQLite as the database,
and exposes the web interface through a public ngrok URL.

---

## 📋 What this notebook does
1. Installs .NET 6.0 SDK on the Colab runtime
2. Clones the PharmaChain repository from GitHub
3. Installs Python helper packages (`pyngrok`, `requests`)
4. Builds the ASP.NET Core app with SQLite support
5. Starts the server and creates a public ngrok tunnel
6. Prints the public URL so you can open the app in a browser

## ⚡ Quick Start
**Run all cells in order** — Runtime → Run all (`Ctrl+F9`)

---

## 🔑 Prerequisites
You need a **free ngrok account** to create the public tunnel:
1. Sign up at [ngrok.com](https://ngrok.com) (free)
2. Copy your auth token from [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Paste it in **Cell 4** below (where it says `YOUR_NGROK_TOKEN`)

---
## 🛠️ Cell 1 — Install .NET 6.0 SDK
Downloads and installs the .NET 6.0 SDK using Microsoft's official install script.
This takes ~2 minutes on first run.

In [ ]:
import os, subprocess, sys

DOTNET_ROOT = '/usr/local/dotnet'
os.environ['DOTNET_ROOT'] = DOTNET_ROOT
os.environ['PATH'] = DOTNET_ROOT + ':' + os.environ.get('PATH', '')

# Download the official dotnet-install script
print('Downloading dotnet-install script...')
!wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh
!chmod +x /tmp/dotnet-install.sh

# Install .NET 6.0 LTS
print('Installing .NET 6.0 SDK (this may take ~2 minutes)...')
!bash /tmp/dotnet-install.sh --channel 6.0 --install-dir /usr/local/dotnet --quiet

# Verify installation
result = subprocess.run(['/usr/local/dotnet/dotnet', '--version'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ .NET SDK installed: {result.stdout.strip()}')
else:
    print('❌ Installation failed:', result.stderr)
    sys.exit(1)

---
## 📦 Cell 2 — Install Python Packages
Installs `pyngrok` for the public tunnel and `requests` for API testing.

In [ ]:
print('Installing Python packages...')
!pip install -q pyngrok requests
print('✅ Python packages installed')

---
## 📂 Cell 3 — Clone PharmaChain Repository
Clones the project from GitHub. If you already uploaded the project files,
skip this cell and set `PROJECT_DIR` to your upload path instead.

In [ ]:
import os

# ── Configure your GitHub repo URL here ───────────────────────────────────
GITHUB_REPO = 'https://github.com/ayajamal1322003-eng/PharmaChain.git'
PROJECT_DIR = '/content/PharmaChain'
# ──────────────────────────────────────────────────────────────────────────

if os.path.exists(PROJECT_DIR):
    print(f'Project directory already exists at {PROJECT_DIR}')
    print('Pulling latest changes...')
    !cd {PROJECT_DIR} && git pull
else:
    print(f'Cloning from {GITHUB_REPO}...')
    !git clone {GITHUB_REPO} {PROJECT_DIR}

# Verify key files exist
required = ['Program.cs', 'PharmaChain.csproj', 'appsettings.Colab.json']
missing  = [f for f in required if not os.path.exists(f'{PROJECT_DIR}/{f}')]

if missing:
    print(f'❌ Missing files: {missing}')
    print('   Make sure you have the latest version of the repo with Colab support.')
else:
    print(f'✅ Project ready at {PROJECT_DIR}')
    print('   Files found:', required)

---
## 🔑 Cell 4 — Configure ngrok Auth Token
**You must do this before launching.**  
Get your free token at [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

In [ ]:
# ── Paste your ngrok auth token here ──────────────────────────────────────
NGROK_AUTH_TOKEN = 'YOUR_NGROK_TOKEN'
# ──────────────────────────────────────────────────────────────────────────

# Optional: set your Anthropic API key to enable AI-powered QR tokens
# Leave empty to use the secure cryptographic fallback (fully functional)
ANTHROPIC_API_KEY = ''

if NGROK_AUTH_TOKEN == 'YOUR_NGROK_TOKEN':
    raise ValueError(
        '\n\n❌ Please replace YOUR_NGROK_TOKEN with your actual ngrok auth token.\n'
        '   Get it free at: https://dashboard.ngrok.com/get-started/your-authtoken\n'
    )

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('✅ ngrok auth token configured')

---
## 🏗️ Cell 5 — Build the Project
Restores NuGet packages and compiles the ASP.NET Core application.
First build takes ~3-5 minutes; subsequent runs are faster.

In [ ]:
import subprocess, os

DOTNET  = '/usr/local/dotnet/dotnet'
PROJECT = '/content/PharmaChain'
PUBLISH = '/content/pharmachain-app'

os.environ['DOTNET_ROOT'] = '/usr/local/dotnet'
os.environ['PATH']        = '/usr/local/dotnet:' + os.environ.get('PATH', '')
# Suppress .NET telemetry
os.environ['DOTNET_CLI_TELEMETRY_OPTOUT'] = '1'
os.environ['DOTNET_NOLOGO']               = '1'

def run(cmd, cwd=PROJECT, label=''):
    print(f'⏳ {label}...')
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                            env=os.environ.copy())
    if result.returncode != 0:
        print('STDOUT:', result.stdout[-3000:] if result.stdout else '(empty)')
        print('STDERR:', result.stderr[-3000:] if result.stderr else '(empty)')
        raise RuntimeError(f'{label} failed (exit {result.returncode})')
    return result.stdout

# Step 1: Restore NuGet packages
run([DOTNET, 'restore', 'PharmaChain.csproj'], label='Restoring NuGet packages')
print('   ✅ Packages restored')

# Step 2: Publish (Release build — self-contained output bundle)
run([DOTNET, 'publish', 'PharmaChain.csproj',
     '-c', 'Release',
     '-o', PUBLISH,
     '/p:UseAppHost=false',
     '--nologo'],
    label='Building & publishing (Release)')
print('   ✅ Build complete')

# Copy static files and Colab appsettings to publish directory
import shutil
for item in ['wwwroot', 'blockchain_chain.json', 'appsettings.Colab.json']:
    src = f'{PROJECT}/{item}'
    dst = f'{PUBLISH}/{item}'
    if os.path.exists(src):
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

print(f'✅ App published to {PUBLISH}')
print('   Ready to launch!')

---
## 🚀 Cell 6 — Launch PharmaChain + Create Public URL
Starts the .NET server on port 5000 and opens an ngrok tunnel.
**The public URL is printed at the end of this cell.**

In [ ]:
import subprocess, os, time, threading
from pyngrok import ngrok

DOTNET  = '/usr/local/dotnet/dotnet'
PUBLISH = '/content/pharmachain-app'
PORT    = 5000

# Kill any previously running instance
!pkill -f 'PharmaChain.dll' 2>/dev/null || true
time.sleep(1)

# Close any existing ngrok tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Open ngrok tunnel first so we know the public URL before starting the app
print(f'Opening ngrok tunnel on port {PORT}...')
public_url = ngrok.connect(PORT, proto='http').public_url
print(f'   Tunnel ready: {public_url}')

# Build environment for the .NET process
env = os.environ.copy()
env.update({
    'DOTNET_ROOT':                    '/usr/local/dotnet',
    'ASPNETCORE_ENVIRONMENT':         'Colab',          # loads appsettings.Colab.json
    'ASPNETCORE_URLS':                f'http://0.0.0.0:{PORT}',
    'DATABASE_PROVIDER':              'Sqlite',
    # Override BaseUrl so QR codes embed the public ngrok URL
    'App__BaseUrl':                   public_url,
    'DOTNET_CLI_TELEMETRY_OPTOUT':    '1',
    'DOTNET_NOLOGO':                  '1',
})

# Inject optional Anthropic API key
if ANTHROPIC_API_KEY:
    env['Anthropic__ApiKey'] = ANTHROPIC_API_KEY

# Start the .NET process in the background
log_path = '/content/pharmachain.log'
with open(log_path, 'w') as log_file:
    proc = subprocess.Popen(
        [DOTNET, f'{PUBLISH}/PharmaChain.dll'],
        cwd=PUBLISH,
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

# Wait for the server to start (polls the log for the ready signal)
print('Starting PharmaChain server', end='')
ready = False
for _ in range(60):          # wait up to 60 seconds
    time.sleep(1)
    print('.', end='', flush=True)
    if os.path.exists(log_path):
        with open(log_path) as f:
            log = f.read()
        if 'Now listening on' in log or 'Application started' in log:
            ready = True
            break
        if 'Unhandled exception' in log or 'FATAL' in log:
            print('\n❌ Server crashed! Showing last log lines:')
            print('\n'.join(log.splitlines()[-20:]))
            break

print()  # newline after dots

if ready:
    print()
    print('=' * 60)
    print('  🚀 PharmaChain is running!')
    print('=' * 60)
    print(f'  🌐 Login page    : {public_url}/login.html')
    print(f'  📊 Dashboard     : {public_url}/dashboard.html')
    print(f'  🔧 Swagger API   : {public_url}/swagger')
    print(f'  🔍 Verify QR     : {public_url}/verify.html')
    print(f'  ⛓️  Blockchain    : {public_url}/blockchain.html')
    print('=' * 60)
    print()
    print('  Default test accounts (register them first via /api/auth/register):')
    print('  Role options: Factory | Distributor | Pharmacy | Customer | Admin')
    print()
    print(f'  Server PID: {proc.pid}  |  Log: {log_path}')
    print('  To stop: !pkill -f PharmaChain.dll')
else:
    print('⚠️  Server did not report ready within 60 s.')
    print('   Check the log:')
    if os.path.exists(log_path):
        with open(log_path) as f:
            print('\n'.join(f.read().splitlines()[-30:]))

---
## 🧪 Cell 7 — Smoke Test (Optional)
Registers a test user, logs in, and adds a sample drug to verify the API works end-to-end.

In [ ]:
import requests, json

BASE = f'http://localhost:5000'   # internal — test server-side

# ── 1. Register a Factory user ─────────────────────────────────────────────
r = requests.post(f'{BASE}/api/auth/register', json={
    'username': 'factory_demo',
    'password': 'Demo1234!',
    'role':     'Factory'
})
if r.status_code == 200:
    print('✅ Registered factory_demo')
elif r.status_code == 400 and 'already' in r.text.lower():
    print('ℹ️  factory_demo already exists — continuing')
else:
    print(f'⚠️  Register: {r.status_code} {r.text}')

# ── 2. Login ───────────────────────────────────────────────────────────────
r = requests.post(f'{BASE}/api/auth/login', json={
    'username': 'factory_demo',
    'password': 'Demo1234!'
})
if r.status_code != 200:
    print(f'❌ Login failed: {r.status_code} {r.text}')
else:
    token = r.json().get('token', '')
    print(f'✅ Login successful. JWT: {token[:40]}...')

    headers = {'Authorization': f'Bearer {token}'}

    # ── 3. Add a sample drug ───────────────────────────────────────────────
    r = requests.post(f'{BASE}/api/drugs', headers=headers, json={
        'name':         'Paracetamol 500mg',
        'batchNumber':  'BATCH-DEMO-001',
        'expiryDate':   '2027-12-31',
        'manufacturer': 'Demo Pharma Co.',
        'quantity':     1000
    })
    if r.status_code == 200:
        drug = r.json()
        print(f'✅ Drug added — ID: {drug.get("id")}, Token: {drug.get("aiToken", "N/A")[:16]}...')
    else:
        print(f'⚠️  Add drug: {r.status_code} {r.text}')

    # ── 4. List drugs ──────────────────────────────────────────────────────
    r = requests.get(f'{BASE}/api/drugs', headers=headers)
    drugs = r.json()
    print(f'✅ Drug count in database: {len(drugs)}')

print()
print('🎉 Smoke test complete. Open the login page and start exploring!')
print(f'   URL: {public_url}/login.html')

---
## 📋 Cell 8 — View Server Logs (Optional)
Shows the last 40 lines of the server log for debugging.

In [ ]:
log_path = '/content/pharmachain.log'

if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(f'--- Last {min(40, len(lines))} log lines ---')
    print(''.join(lines[-40:]))
else:
    print('No log file found. Run Cell 6 first.')

---
## 🛑 Cell 9 — Stop the Server (Optional)

In [ ]:
import os
from pyngrok import ngrok

# Kill the .NET process
!pkill -f 'PharmaChain.dll' 2>/dev/null || true

# Close all ngrok tunnels
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)
ngrok.kill()

print('✅ Server and tunnels stopped.')

---
## 📖 Usage Guide

### 👥 User Roles & Supply Chain Flow
```
Factory  →  Distributor  →  Pharmacy  →  Customer
```
Each role can only transfer to the next role in the chain.

### 🚀 Getting Started
1. Open `{public_url}/login.html` in your browser
2. Register accounts for each role you want to test:
   - Go to Swagger (`/swagger`) → `POST /api/auth/register`
   - Or use the test script in **Cell 7** above
3. Log in as **Factory** → Add a drug → Generate a QR code
4. Log in as **Distributor** → Transfer the drug → Scan the QR
5. Continue through Pharmacy → Customer to complete the chain

### 🔑 API Endpoints
| Endpoint | Method | Description |
|----------|--------|-------------|
| `/api/auth/register` | POST | Create user account |
| `/api/auth/login` | POST | Get JWT token |
| `/api/drugs` | GET/POST | List / add drugs |
| `/api/qr/{drugId}` | GET | Generate QR code |
| `/api/verify` | POST | Verify QR authenticity |
| `/api/transaction/transfer` | POST | Transfer drug (blockchain) |
| `/api/transaction/chain` | GET | View full blockchain |
| `/api/audit` | GET | View audit log |

### 💾 Data Persistence
- **Database**: SQLite file at `/content/PharmaChain/pharmachain.db`
- **Blockchain backup**: `/content/pharmachain-app/blockchain_chain.json`
- ⚠️ Colab resets its filesystem when the session ends. Download these files if you want to preserve data.

### 🔐 Security Features
- JWT Bearer authentication with role-based access control
- Login throttling (5 attempts → 5-minute lockout)
- HMAC-SHA256 signed QR codes with AI-generated tokens
- Proof-of-Work blockchain (SHA-256, difficulty=2)
- Attack detection: signature forgery, date tampering, replay attacks

### 🌐 Language
The UI supports both **Arabic (RTL)** and **English (LTR)** — toggle using the language button in the top-right corner of any page.

---
*PharmaChain — Blockchain-Secured Pharmaceutical Supply Chain*